In [1]:
import sys
import cv2
import mediapipe as mp
from PyQt5.QtWidgets import QApplication, QLabel, QWidget, QVBoxLayout
from PyQt5.QtGui import QImage, QPixmap
from PyQt5.QtCore import QTimer

# Mediapipe Gesichtserkennung initialisieren
mp_face_detection = mp.solutions.face_detection

class FaceDetectorApp(QWidget):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Face Detection")
        self.setGeometry(100, 100, 800, 600)  # Fenstergröße setzen

        # Kamera starten
        self.cap = cv2.VideoCapture(0)
        if not self.cap.isOpened():
            print("⚠️ Fehler: Kamera kann nicht geöffnet werden!")

        # QLabel für die Live-Kamera
        self.label = QLabel(self)
        layout = QVBoxLayout()
        layout.addWidget(self.label)
        self.setLayout(layout)

        # Gesichtserkennung aktivieren
        self.face_detection = mp_face_detection.FaceDetection(min_detection_confidence=0.5)

        # Timer für das Update der Kamera
        self.timer = QTimer()
        self.timer.timeout.connect(self.update_frame)
        self.timer.start(10)

    def update_frame(self):
        ret, frame = self.cap.read()
        if not ret:
            print("❌ Kein Kamerabild erhalten.")
            return
        
        # Bild in RGB umwandeln (Mediapipe benötigt RGB)
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = self.face_detection.process(frame_rgb)

        # Gesichter markieren
        if results.detections:
            print("✅ Gesicht erkannt!")
            for detection in results.detections:
                bboxC = detection.location_data.relative_bounding_box
                h, w, _ = frame.shape
                bbox = (
                    int(bboxC.xmin * w),
                    int(bboxC.ymin * h),
                    int(bboxC.width * w),
                    int(bboxC.height * h)
                )
                print(f"🟢 Bounding Box: {bbox}")
                cv2.rectangle(frame_rgb, bbox, (0, 255, 0), 2)  # Bounding Box zeichnen

        # OpenCV Frame in QImage konvertieren
        h, w, ch = frame_rgb.shape
        bytes_per_line = ch * w
        q_img = QImage(frame_rgb.data, w, h, bytes_per_line, QImage.Format_RGB888)
        pixmap = QPixmap.fromImage(q_img)

        # QLabel mit neuem Bild aktualisieren
        self.label.setPixmap(pixmap)

    def closeEvent(self, event):
        self.cap.release()
        event.accept()

# Hauptprogramm
if __name__ == "__main__":
    app = QApplication(sys.argv)
    window = FaceDetectorApp()
    window.show()
    sys.exit(app.exec_())



ModuleNotFoundError: No module named 'PyQt5'